### <font color='#4285f4'>Overview</font>

**Conversational Analytics (CAI) — Expert Agentic Skills Lab**

This notebook teaches you to build and interact with the BigQuery **Conversational Analytics API** (`geminidataanalytics`) the way a Customer Engineer does in the field:

1. The **three interaction modes**: stateful conversation, stateless-with-agent, stateless inline.
2. Building a **rich context** from a curated table list: system instruction, business glossary, and golden (example) queries.
3. Understanding the **response stream** (answer vs. thoughts vs. follow-ups), and idempotent agent lifecycle.

**You can run this whole notebook on `bigquery-public-data`** — no lab deployment required. The messy `l400_agentic_da` ERP schema is the *worked example* you graduate to (Part 4).

> **This notebook is an optional tutorial** for the reusable `conversational-analytics` skill (`.agents/skills/conversational-analytics/`). The skill is the product; this just demonstrates it. To use the skill directly on any machine, work from its folder with `uv` (see that folder's `README`/`SKILL.md`). The Initialize cell below finds the skill on disk, so run this from a repo checkout.

Author: Google Cloud Data Analytics CE enablement.

### <font color='#4285f4'>License</font>

```
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#     https://www.apache.org/licenses/LICENSE-2.0
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
```

### <font color='#4285f4'>Pip installs</font>

In [1]:
import sys
# On a fresh Colab runtime this installs the skill's deps. If you're running in a
# prepared environment that already has them, pip may be absent — that's fine,
# just skip this cell.
try:
    import google.cloud.geminidataanalytics, google.auth  # noqa: F401
    print('Dependencies already present — skipping install.')
except ImportError:
    !{sys.executable} -m pip install -q 'google-cloud-geminidataanalytics>=0.11.0' google-auth requests proto-plus

Dependencies already present — skipping install.


### <font color='#4285f4'>Initialize</font>

In [2]:
import os, sys
import google.auth

# Find the conversational-analytics skill's scripts/ dir. This notebook is a
# TUTORIAL for the skill — it needs the skill folder on disk. We check the common
# launch locations (repo checkout, running from colab-enterprise/, cwd, or an
# env override) rather than assuming one. The real way to use the skill is from
# its own folder via uv (see that folder's README) — this just makes the tutorial
# import it wherever you launched Jupyter from.
_CANDIDATES = [
    os.environ.get('CA_SKILL_SCRIPTS'),                                  # explicit override
    os.path.join('..', '.agents', 'skills', 'conversational-analytics', 'scripts'),  # from colab-enterprise/
    os.path.join('.agents', 'skills', 'conversational-analytics', 'scripts'),        # from repo root
    os.path.join(os.path.expanduser('~'), 'git', 'l400-agentic-da', '.agents',
                 'skills', 'conversational-analytics', 'scripts'),        # common clone location
]
SKILL_SCRIPTS = next((os.path.abspath(p) for p in _CANDIDATES
                      if p and os.path.isdir(p)), None)
if SKILL_SCRIPTS is None:
    raise RuntimeError(
        "Could not find the conversational-analytics skill's scripts/ dir.\n"
        "These notebooks are tutorials for that skill and need it on disk.\n"
        "Fix: clone the repo and launch Jupyter from it, or set CA_SKILL_SCRIPTS\n"
        "to <repo>/.agents/skills/conversational-analytics/scripts.\n"
        "Best experience: run from the skill folder's uv env (see its README).")
sys.path.insert(0, SKILL_SCRIPTS)
import ca   # the facade: SDK by default, ca.set_backend('rest') to switch

PROJECT = os.environ.get('CA_PROJECT') or google.auth.default()[1]
LOCATION = 'global'
print('skill:', SKILL_SCRIPTS)
print('project:', PROJECT, '| backend:', ca.get_backend())

skill: /home/user/git/l400-agentic-da/.agents/skills/conversational-analytics/scripts
project: statmike-mlops-349915 | backend: sdk


### <font color='#4285f4'>The skill in one glance</font>

The `conversational-analytics` skill gives you one import (`ca`) covering every CAI operation, in two mirrored implementations (SDK default; `ca.set_backend('rest')` for REST-only environments — the REST path never imports the SDK). Functional areas:

| Area | Functions |
|---|---|
| chat | `ca.chat(..., context_mode=)` over all 3 modes |
| agents | `create_data_agent`, `create_or_update_data_agent`, `update_data_agent`, `get`/`list`/`delete` |
| conversations | `create_conversation`, `get_or_create_conversation`, `get_conversation`, `list_messages` |
| context | `build_table_references`, `build_glossary_terms`, `build_example_queries`, `build_context` |
| responses | `parse_stream`, `find_messages` |
| ADK (optional) | `to_tool_json` (see references/) |

Every call returns `{status, tool_name, messages, results}` — check `status` before reading `results`.

The skill **consumes** a curated table list + context; in the lab those are authored by the `ca-generate-*` prompt flow (see the skills README). The skill's job is to turn them into a Context and deploy/chat.

### <font color='#4285f4'>Part 1 — Stateless inline chat (the simplest path)</font>

Inline mode creates **no stored agent**: you pass the full context with each request. Great for embedded tools and dynamic routing. We use the public `penguins` table.

In [3]:
tr = ca.build_table_references(['bigquery-public-data.ml_datasets.penguins'])
ctx = ca.build_context('You are a helpful data analyst.',
                       tr['results']['datasource_references'], enable_python=False)

resp = ca.chat(PROJECT, LOCATION, 'What is the average body mass by island?',
               'inline', inline_context=ctx['results']['context'])

ans = resp['results']['answer']
print('ANSWER:\n', ans['text'])
print('\nSQL the agent ran:\n', ans['generated_sql'])

ANSWER:
 Here is the average body mass of penguins calculated for each island.
### Key Insights
* **Biscoe Island** has the highest average body mass at **4,716.02g** (approximately 4.72 kg), which is significantly higher than the other two islands.
* **Dream Island** and **Torgersen Island** have very similar average penguin body masses, at **3,712.90g** and **3,706.37g** respectively.

SQL the agent ran:
 SELECT
  penguins.island,
  AVG(penguins.body_mass_g) AS average_body_mass_g
FROM
  `bigquery-public-data.ml_datasets.penguins` AS penguins
GROUP BY
  penguins.island
ORDER BY
  average_body_mass_g DESC;


#### Understanding the response
The API returns many messages in non-deterministic order. `parse_stream` (already applied inside `chat`) buckets them by `textType`. Note the answer is **not** the last text message — that's usually the follow-up questions.

In [4]:
# parse_stream (applied inside chat) buckets the messages. Note how the answer,
# reasoning, and follow-ups are cleanly separated:
print('answer (first line):', (ans['text'] or '').split(chr(10))[0][:80])
print('thoughts     :', len(ans['thoughts']), 'reasoning/progress messages')
print('followups    :', len(ans['followups']), 'suggested next questions')
print('data rows    :', len(ans['data']) if ans['data'] else 0)
print('has chart    :', ans['has_chart'])

# Locate specific raw messages when you need them (e.g. to render a chart):
raw = resp['results']['raw']
print('\nFINAL_RESPONSE messages:', len(ca.find_messages(raw, 'text', text_type='FINAL_RESPONSE')))
print('FOLLOWUP messages      :', len(ca.find_messages(raw, 'text', text_type='FOLLOWUP_QUESTIONS')))

answer (first line): Here is the average body mass of penguins calculated for each island.
thoughts     : 4 reasoning/progress messages
followups    : 1 suggested next questions
data rows    : 3
has chart    : False

FINAL_RESPONSE messages: 2
FOLLOWUP messages      : 1


### <font color='#4285f4'>Part 2 — Stateful agent (durable, multi-turn)</font>

A stored **data agent** + a **conversation**: the server keeps history, so follow-ups just work. `create_or_update_data_agent` is idempotent — safe to re-run this cell.

In [5]:
# A stable id so re-runs reuse the same agent (idempotent). If you ever hard-delete
# it, the id is soft-deleted/reserved for ~30 days — bump the suffix for a fresh one.
AGENT_ID = 'cai-lab-penguin-agent'
CONV_ID = 'cai-lab-penguin-conv'

agent = ca.create_or_update_data_agent(
    PROJECT, LOCATION, AGENT_ID, ctx['results']['context'],
    display_name='Lab Penguin Agent', labels={'origin': 'cai-lab'})
agent_name = f'projects/{PROJECT}/locations/{LOCATION}/dataAgents/{AGENT_ID}'
print('agent:', agent['status'], '| created:', agent['results'].get('created'))
if agent['status'] != 'success':
    print('  ->', agent['messages'][0])   # e.g. soft-deleted id: bump the suffix above

# get_or_create is idempotent -> safe to re-run this cell.
conv = ca.get_or_create_conversation(PROJECT, LOCATION, CONV_ID, AGENT_ID)
conv_name = conv['results']['name']
print('conversation:', conv['status'], '| created:', conv['results'].get('created'))

for q in ['How many penguins per island?', 'Which of those has the heaviest on average?']:
    r = ca.chat(PROJECT, LOCATION, q, 'conversation',
                agent_name=agent_name, conversation_name=conv_name)
    if r['status'] == 'success':
        print(f'\nQ: {q}\nA: {(r["results"]["answer"]["text"] or "")[:200]}')
    else:
        print(f'\nQ: {q}\n[error] {r["messages"][0]}')

agent: success | created: False
conversation: success | created: False

Q: How many penguins per island?
A: Here is the total number of penguins recorded on each island in the dataset:
### Key Insights

- **Biscoe Island** is the most populated island in the dataset, containing **168 penguins** (over half o

Q: Which of those has the heaviest on average?
A: Based on the average body mass of the penguins, **Biscoe Island** has the heaviest penguins on average.
### Key Insights

- **Biscoe Island** has significantly heavier penguins compared to the other t


### <font color='#4285f4'>Part 3 — Rich context: glossary + golden queries</font>

The real skill is *authoring context*. On a cryptic legacy schema, the agent needs your business glossary (what terms mean) and golden queries (verified SQL for hard metrics). Here's the shape — in the lab these come from the `l400-agentic-data-analytics` artifacts (see Part 6).

In [6]:
glossary = ca.build_glossary_terms([
    {'display_name': 'Heavy Penguin', 'description': 'A penguin with body_mass_g > 5000.'},
])
golden = ca.build_example_queries([
    {'natural_language_question': 'How many heavy penguins are there?',
     'sql_query': 'SELECT COUNT(*) FROM `bigquery-public-data.ml_datasets.penguins` WHERE body_mass_g > 5000'},
])
rich_ctx = ca.build_context(
    'You are a penguin analyst. Use the glossary for domain terms.',
    tr['results']['datasource_references'],
    glossary_terms=glossary['results']['glossary_terms'],
    example_queries=golden['results']['example_queries'])

r = ca.chat(PROJECT, LOCATION, 'How many heavy penguins are there?', 'inline',
            inline_context=rich_ctx['results']['context'])
print(r['results']['answer']['text'])

There are 61 penguins classified as heavy. 

This is determined based on the definition of a heavy penguin as having a body mass of more than 5,000 grams. I retrieved this count by running the verified heavy penguins example query.
### Key Observations
Out of the entire dataset, there are 61 penguins that weigh more than 5,000 grams. This represents the heavier subset of the penguin population.


### <font color='#4285f4'>Part 4 — Graduate to the `l400-agentic-data-analytics` lab</font>

Everything above ran on public data. The full lab uses the deliberately messy `l400_agentic_da` ERP schema (~20 cryptic tables). There, the CA agent definition (system prompt, table list, golden queries) is authored by the `ca-generate-*` prompt flow (`docs/requirements/Conversational-Analytics-Best-Practices/ca-generate-*.md` → `okf-staging/conversational-analytics/*`), then this skill deploys it. See `.agents/skills/README.md` for the full pipeline.

To deploy `l400-agentic-data-analytics` itself, use a **dedicated GCP project** — the standard `deploy.sh` creates a fresh project via Terraform, so full teardown is `terraform destroy` (or deleting that project). See `.agents/skills/conversational-analytics/evals/deploy-checklist.md`. (Note: `clean-up.sh` only removes *local* Terraform state, not cloud resources.)

### <font color='#4285f4'>Clean Up</font>
By default the agent is **left in place** so you can re-run the notebook freely. Set `DELETE_AGENT = True` only if you want to remove it — note that deletes are *soft* (the id is reserved ~30 days).

In [7]:
# By default we DO NOT delete the agent, so you can re-run this notebook freely
# (create_or_update + get_or_create make re-runs idempotent).
#
# IMPORTANT: deleting a data agent SOFT-deletes it. The id stays reserved (~30
# days until purge) and there is no undelete API, so you cannot recreate an agent
# with the same id until it purges. Only delete when you truly want that id gone.
DELETE_AGENT = False

if DELETE_AGENT:
    print(ca.delete_data_agent(PROJECT, LOCATION, AGENT_ID)['status'])
else:
    print(f"Left agent '{AGENT_ID}' in place (re-runnable). Set DELETE_AGENT=True to remove it.")

Left agent 'cai-lab-penguin-agent' in place (re-runnable). Set DELETE_AGENT=True to remove it.


### <font color='#4285f4'>Reference Links</font>

- Skill: `.agents/skills/conversational-analytics/` (SKILL.md + references/ + evals/)
- [Build a data agent (Python SDK)](https://docs.cloud.google.com/gemini/docs/conversational-analytics-api/build-agent-sdk)
- [Build a data agent (HTTP)](https://docs.cloud.google.com/gemini/data-agents/conversational-analytics-api/build-agent-http)
- [Define data agent context for BigQuery](https://docs.cloud.google.com/gemini/docs/conversational-analytics-api/data-agent-authored-context-bq)
- Worked examples & tests: `.agents/skills/conversational-analytics/{evals,tests}/`